# Additional End of week Exercise - Week 2
## TechMentor AI: Full Prototype Technical Question & Answer Assistant

Welcome to the Week 2 Capstone Project! In this notebook, we build on the Week 1 Technical Question/Answerer by integrating everything learned across Week 2:
- **Gradio UI**: Interactive, modern interface built with `gr.Blocks`
- **Streaming Responses**: Real-time token streaming using generator yields for responsive UX
- **System Prompt Expertise**: Multi-tier technical personas (Principal Software Engineer, Performance Architect, Beginner Mentor)
- **Model Switching**: Dynamic routing between **Groq** (GPT-OSS 120B / Qwen 27B), **OpenAI / OpenRouter** (GPT-4o-mini), and **Ollama** (Llama 3.2)
- **Tool Calling (Bonus Points!)**: An isolated **Python Sandbox Code Runner** & **AST Syntax Inspector** so the assistant can test and verify code execution live
- **Audio Input & Output (Bold Bonus!)**: Speak questions via **Whisper (STT)** and listen to explanations via **Text-to-Speech (TTS)**


### 1. Imports & Environment Setup
We load our environment keys from `.env` and verify available dependencies (`gradio`, `openai`, `groq`, `gTTS`).


In [1]:
import os
import sys
import io
import ast
import json
import tempfile
import traceback
from typing import Generator, Tuple, Optional, List, Dict, Any
from dotenv import load_dotenv

# Load API keys from .env
load_dotenv(override=True)

import gradio as gr
from openai import OpenAI

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False

try:
    from gtts import gTTS
    GTTS_AVAILABLE = True
except ImportError:
    GTTS_AVAILABLE = False

print(f"Groq library available: {GROQ_AVAILABLE}")
print(f"gTTS library available: {GTTS_AVAILABLE}")
print(f"Gradio version: {gr.__version__}")


Groq library available: True
gTTS library available: True
Gradio version: 6.29.1


### 2. Multi-Model Client Routing
We initialize our providers (Groq, OpenAI / OpenRouter, Ollama) and configure model specifications so users can switch models on the fly.


In [2]:
openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

# Initialize API clients
groq_client = Groq(api_key=groq_api_key) if (GROQ_AVAILABLE and groq_api_key) else None
openai_client = OpenAI(api_key=openai_api_key, base_url=openai_base_url) if openai_api_key else None
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODEL_OPTIONS = {
    "Groq: OpenAI GPT-OSS 120B (Ultra-fast, Free)": {
        "provider": "groq",
        "model_id": "openai/gpt-oss-120b",
        "supports_tools": True
    },
    "Groq: Qwen 3.8 27B (Fast & Accurate)": {
        "provider": "groq",
        "model_id": "qwen/qwen3.8-27b",
        "supports_tools": True
    },
    "OpenRouter: Z.ai: GLM 5.3 Flash": {
        "provider": "openai",
        "model_id": "z-ai/glm-5.3-flash" if "openrouter" not in openai_base_url else "z-ai/glm-5.3-flash",
        "supports_tools": True
    },
    "OpenRouter: NVIDIA: Nemotron 3 Ultrah": {
        "provider": "openai",
        "model_id": "nvidia/nemotron-3-ultra-550b-a55b:free" if "openrouter" not in openai_base_url else "nvidia/nemotron-3-ultra-550b-a55b:free",
        "supports_tools": True
    },
    "Ollama: Llama 3.2 (Local Machine)": {
        "provider": "ollama",
        "model_id": "llama3.2",
        "supports_tools": False
    }
}

print("Available Models:", list(MODEL_OPTIONS.keys()))


Available Models: ['Groq: OpenAI GPT-OSS 120B (Ultra-fast, Free)', 'Groq: Qwen 3.8 27B (Fast & Accurate)', 'OpenRouter: Z.ai: GLM 5.3 Flash', 'OpenRouter: NVIDIA: Nemotron 3 Ultrah', 'Ollama: Llama 3.2 (Local Machine)']


### 3. System Prompting for Technical Expertise
A system prompt transforms a generic model into an authoritative technical mentor. We define specialized personas:
- **Principal Software Engineer**: High-level overview, step-by-step mechanical syntax breakdown, edge cases, time/space complexity, and idiomatic alternatives.
- **Beginner-Friendly Mentor**: Plain-English analogies, gentle explanations, no unexplained jargon.
- **Refactoring & Performance Architect**: Memory footprint, generator efficiency, and PEP 8 best practices.


In [3]:
SYSTEM_PERSONAS = {
    "Principal Software Engineer": (
        "You are a Principal Software Engineer and elite Computer Science Educator with 20+ years of experience. "
        "When explaining technical questions or code snippets:\n"
        "1. Provide a succinct 1-2 sentence high-level overview of what the code does.\n"
        "2. Break down the mechanical syntax step-by-step (e.g. comprehensions, generators, memory layout, AST).\n"
        "3. Highlight potential edge cases, failure modes, and time/space complexity (Big-O).\n"
        "4. Provide a cleaner, more Pythonic or production-ready alternative if applicable.\n"
        "5. If testing the code helps clarify the explanation, you may use the run_python_code tool to demonstrate actual output."
    ),
    "Beginner-Friendly Mentor": (
        "You are an encouraging, patient Python mentor for junior developers. "
        "Avoid unnecessary jargon. When explaining code, use relatable analogies, explain tricky concepts in simple terms, "
        "and provide annotated examples showing exactly what is happening under the hood."
    ),
    "Refactoring & Performance Architect": (
        "You are a Senior Performance Architect and Code Review Specialist. "
        "Focus on code smells, PEP 8 idiomatic Python, memory footprint, time complexity, and generator efficiency. "
        "Suggest modern Python 3.10+ alternatives and benchmark-friendly patterns."
    )
}


### 4. Tool Execution: Python Sandbox & AST Parser (Bonus Points!)
We give the model the ability to run code snippets in a local execution sandbox and inspect AST structures.
When the user asks about tricky syntax (like `yield from {book.get('author') for book in books if book.get('author')}`), the model can run the code against mock data to verify output!


In [4]:
def run_python_code(code: str) -> str:
    """Safely executes Python code in a local sandbox scope and captures stdout."""
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    captured_io = io.StringIO()
    sys.stdout = captured_io
    sys.stderr = captured_io
    try:
        scope: Dict[str, Any] = {}
        compiled = compile(code, "<sandbox>", "exec")
        exec(compiled, scope)
        output = captured_io.getvalue().strip()
        if not output:
            output = "Code ran successfully with no stdout output."
        return output
    except Exception as e:
        return f"Execution Error: {e}\n{traceback.format_exc()}"
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr

def inspect_python_syntax(code: str) -> str:
    """Parses Python code and dumps its AST structure to inspect nodes and tokens."""
    try:
        tree = ast.parse(code)
        return ast.dump(tree, indent=2)
    except SyntaxError as e:
        return f"Syntax Error at line {e.lineno}, col {e.offset}: {e.msg}"

TOOLS_SPEC = [
    {
        "type": "function",
        "function": {
            "name": "run_python_code",
            "description": "Execute Python code in an isolated scope to verify behavior, test output, or evaluate an example. Always include print() to show output.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python source code to execute."
                    }
                },
                "required": ["code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "inspect_python_syntax",
            "description": "Parse Python source code into an Abstract Syntax Tree (AST) to inspect its grammatical syntax and node types.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python source code snippet to parse."
                    }
                },
                "required": ["code"]
            }
        }
    }
]

def execute_tool_call(tool_name: str, arguments: dict) -> str:
    if tool_name == "run_python_code":
        return run_python_code(arguments.get("code", ""))
    elif tool_name == "inspect_python_syntax":
        return inspect_python_syntax(arguments.get("code", ""))
    return f"Unknown tool: {tool_name}"


### 5. Multi-Modal Audio: Whisper STT & TTS (Bold Bonus!)
- **Speech-to-Text**: Groq Whisper (`whisper-large-v3-turbo`) or OpenAI Whisper (`whisper-1`) transcribes microphone audio.
- **Text-to-Speech**: gTTS or OpenAI TTS converts the technical explanation into high quality spoken voice audio.


In [5]:
def transcribe_audio(audio_path: Optional[str]) -> str:
    """Transcribes voice audio to text using Whisper."""
    if not audio_path:
        return ""
    try:
        with open(audio_path, "rb") as audio_file:
            if groq_client:
                transcription = groq_client.audio.transcriptions.create(
                    model="whisper-large-v3-turbo",
                    file=audio_file
                )
                return transcription.text.strip()
            elif openai_client:
                transcription = openai_client.audio.transcriptions.create(
                    model="whisper-1",
                    file=audio_file
                )
                return transcription.text.strip()
            else:
                return "Audio transcription unavailable: No Groq or OpenAI key configured."
    except Exception as e:
        return f"[Audio transcription error: {str(e)}]"

def synthesize_speech(text: str) -> Optional[str]:
    """Generates spoken audio from text using gTTS or OpenAI TTS."""
    if not text or not text.strip():
        return None
    
    clean_text = "\n".join([line for line in text.split("\n") if not line.strip().startswith("```")])
    words = clean_text.split()
    if len(words) > 120:
        clean_text = " ".join(words[:120]) + " ... Full detailed explanation available in text below."

    try:
        temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        temp_path = temp_audio.name
        temp_audio.close()

        if GTTS_AVAILABLE:
            tts = gTTS(text=clean_text, lang="en")
            tts.save(temp_path)
            return temp_path
        elif openai_client:
            response = openai_client.audio.speech.create(
                model="tts-1",
                voice="alloy",
                input=clean_text
            )
            with open(temp_path, "wb") as f:
                f.write(response.content)
            return temp_path
    except Exception as e:
        print(f"TTS error: {e}")
        return None
    return None


### 6. Streaming Callback & Tool Loop
This generator function orchestrates the full pipeline:
1. Audio transcription if voice input was supplied
2. System persona prompt injection
3. Multi-model routing
4. Tool calling execution loop (handling tool call responses)
5. Streaming token generator yielding in real-time to the Gradio Chatbot
6. Audio speech synthesis


In [6]:
def ask_assistant(
    user_question: str,
    audio_file: Optional[str],
    model_choice: str,
    persona_choice: str,
    custom_system_prompt: str,
    enable_tools: bool,
    enable_audio_reply: bool,
    chat_history: List[Dict[str, str]]
) -> Generator[Tuple[List[Dict[str, str]], Optional[str], str], None, None]:
    if audio_file and not user_question.strip():
        transcribed = transcribe_audio(audio_file)
        if transcribed and not transcribed.startswith("["):
            user_question = transcribed
        else:
            user_question = f"(Transcribed: {transcribed})"

    if not user_question.strip():
        yield chat_history, None, "Please type a question or record audio!"
        return

    new_history = list(chat_history)
    new_history.append({"role": "user", "content": user_question})

    if custom_system_prompt.strip():
        system_instruction = custom_system_prompt.strip()
    
    messages: List[Dict[str, Any]] = [{"role": "system", "content": system_instruction}]
    for msg in new_history:
        messages.append({"role": msg["role"], "content": msg["content"]})

    config = MODEL_OPTIONS.get(model_choice, list(MODEL_OPTIONS.values())[0])
    provider = config["provider"]
    model_id = config["model_id"]
    supports_tools = config["supports_tools"] and enable_tools

    client = None
    if provider == "groq" and groq_client:
        client = groq_client
    elif provider == "openai" and openai_client:
        client = openai_client
    elif provider == "ollama":
        client = ollama_client
    else:
        client = groq_client or openai_client or ollama_client

    if client is None:
        new_history.append({"role": "assistant", "content": "⚠️ No LLM client configured."})
        yield new_history, None, "No API client available."
        return

    new_history.append({"role": "assistant", "content": ""})
    yield new_history, None, "Thinking..."

    tool_logs = []
    max_tool_rounds = 2
    round_count = 0

    while round_count < max_tool_rounds:
        round_count += 1
        call_kwargs: Dict[str, Any] = {"model": model_id, "messages": messages}
        if provider == "openai" and "openrouter" in openai_base_url:
            call_kwargs["max_tokens"] = 1000
        if supports_tools:
            call_kwargs["tools"] = TOOLS_SPEC

        try:
            if supports_tools:
                resp = client.chat.completions.create(**call_kwargs)
                choice = resp.choices[0]
                if choice.finish_reason == "tool_calls" and choice.message.tool_calls:
                    messages.append(choice.message)
                    for tc in choice.message.tool_calls:
                        func_name = tc.function.name
                        try:
                            args = json.loads(tc.function.arguments)
                        except Exception:
                            args = {}
                        tool_logs.append(f"🛠️ **Tool Called:** `{func_name}`\n```json\n{json.dumps(args, indent=2)}\n```")
                        yield new_history, None, f"⚙️ Executing tool: `{func_name}`..."

                        result = execute_tool_call(func_name, args)
                        tool_logs.append(f"📊 **Tool Result:**\n```text\n{result}\n```")
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

                    yield new_history, None, "Tool completed. Synthesizing technical answer..."
                    continue
            break
        except Exception as e:
            tool_logs.append(f"*(Tool calling error: {e})*")
            break

    stream_kwargs: Dict[str, Any] = {"model": model_id, "messages": messages, "stream": True}
    if provider == "openai" and "openrouter" in openai_base_url:
        stream_kwargs["max_tokens"] = 1200

    accumulated = ""
    try:
        stream = client.chat.completions.create(**stream_kwargs)
        for chunk in stream:
            delta = chunk.choices[0].delta.content or ""
            accumulated += delta
            new_history[-1]["content"] = accumulated
            yield new_history, None, "Streaming response..."
    except Exception as e:
        accumulated += f"\n\n*(Error during streaming: {e})*"
        new_history[-1]["content"] = accumulated
        yield new_history, None, f"Error: {e}"

    audio_path = None
    if enable_audio_reply:
        yield new_history, None, "🎙️ Generating voice audio response..."
        audio_path = synthesize_speech(accumulated)

    final_tool_log = "\n\n---\n\n".join(tool_logs) if tool_logs else "No tools called for this query."
    yield new_history, audio_path, f"Completed!\n\n{final_tool_log}"


### 7. Gradio Blocks UI & Interactive Launch
Now we assemble the user interface using `gr.Blocks`, with quick example buttons including the Week 1 generator/comprehension question.


In [7]:
def build_interface():
    with gr.Blocks(title="TechMentor AI - Technical Question Answering Prototype") as app:
        gr.Markdown(
            """
            # 🚀 TechMentor AI — Technical Question & Answerer
            ### *Week 2 Prototype: Frontier Models • Streaming • System Expertise • Code Sandbox Tools • Voice I/O*
            """
        )

        with gr.Row():
            # Controls & Inputs
            with gr.Column(scale=4):
                gr.Markdown("### ⚙️ Engine Settings")
                model_dropdown = gr.Dropdown(
                    choices=list(MODEL_OPTIONS.keys()),
                    value=list(MODEL_OPTIONS.keys())[0],
                    label="Model Selector"
                )
                persona_dropdown = gr.Dropdown(
                    choices=list(SYSTEM_PERSONAS.keys()),
                    value="Principal Software Engineer",
                    label="System Persona"
                )
                with gr.Accordion("🛠️ Custom System Prompt", open=False):
                    custom_prompt = gr.Textbox(lines=2, label="Override System Prompt")

                with gr.Row():
                    enable_tools_cb = gr.Checkbox(value=True, label="Enable Tools (Sandbox/AST)")
                    enable_audio_cb = gr.Checkbox(value=True, label="Enable Voice Output (TTS)")

                gr.Markdown("### 🎙️ Audio Input")
                audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Voice Question")

                gr.Markdown("### 💬 Question / Code Snippet")
                question_input = gr.Textbox(
                    lines=4,
                    label="Enter technical question:",
                    placeholder="e.g. Please explain what this code does and why:\nyield from {book.get('author') for book in books if book.get('author')}"
                )

                with gr.Row():
                    submit_btn = gr.Button("🚀 Ask Assistant", variant="primary", scale=2)
                    clear_btn = gr.Button("🗑️ Clear", scale=1)

                gr.Markdown("### 💡 Quick Examples")
                example_1 = gr.Button("📚 Week 1 Question: yield from comprehension")
                example_2 = gr.Button("⚡ Mutable Default Argument Bug")
                example_3 = gr.Button("🔄 Python *args vs **kwargs")

            # Chatbot & Outputs
            with gr.Column(scale=6):
                gr.Markdown("### 🧠 Technical Explanation (Live Streaming)")
                chatbot = gr.Chatbot(label="Conversation", height=500)
                audio_output = gr.Audio(label="🔊 Spoken Response", autoplay=False, type="filepath")
                with gr.Accordion("🔍 Tool Execution Logs", open=True):
                    tool_status = gr.Markdown("Status: Ready.")

        # Event listeners
        inputs = [question_input, audio_input, model_dropdown, persona_dropdown, custom_prompt, enable_tools_cb, enable_audio_cb, chatbot]
        outputs = [chatbot, audio_output, tool_status]

        submit_btn.click(fn=ask_assistant, inputs=inputs, outputs=outputs)
        question_input.submit(fn=ask_assistant, inputs=inputs, outputs=outputs)
        clear_btn.click(fn=lambda: ([], "", None, None, "Status: Cleared."), outputs=[chatbot, question_input, audio_input, audio_output, tool_status])

        example_1.click(
            fn=lambda: "Please explain what this code does and why:\nyield from {book.get('author') for book in books if book.get('author')}\n\nAlso test it with sample book data to verify the output.",
            outputs=question_input
        )
        example_2.click(
            fn=lambda: "Why is using a mutable default argument in Python risky?\n```python\ndef append_to(item, target=[]):\n    target.append(item)\n    return target\n```\nDemonstrate what happens across multiple calls.",
            outputs=question_input
        )
        example_3.click(
            fn=lambda: "Explain how *args and **kwargs work in Python, including tuple/dict unpacking and keyword-only arguments.",
            outputs=question_input
        )

    return app

app = build_interface()
app.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
